# AI工学101 — 第22回

## Random Forest：複数の決定木を組み合わせて強くする

第21回では、**決定木（Decision Tree）**を学びました。

決定木は、

> 「条件分岐をデータから学習する」

という、とても直感的なモデルでした。

ただし、決定木には弱点があります。

**1本の木に判断を任せると、データにかなり敏感になりやすい。**

そこで今日は、

> **「じゃあ、木を1本に任せなければいいじゃん」**

という発想に進みます。

複数の決定木を作り、それぞれの判断を組み合わせる。

これが**アンサンブル学習（Ensemble Learning）**の基本です。

その代表が、

> **Random Forest**

です。

---

# 🎯 今日のゴール

今日は次のことを身につけます。

* アンサンブル学習の基本思想を説明できる
* Random Forestが複数の決定木を組み合わせる仕組みを理解する
* `RandomForestClassifier` を実装できる
* `n_estimators` と `max_depth` の意味を理解する
* 決定木1本とRandom Forestを比較できる
* `feature_importances_` を使って特徴量の重要度を調べられる

---

# 📖 講義：約20分

## 1. 一人の専門家より、複数の専門家

例えば、ある患者について

```text
医師A → 疾患あり

医師B → 疾患なし

医師C → 疾患あり

医師D → 疾患あり

医師E → 疾患なし
```

となったら、

多数決で

```text
疾患あり
```

と判断できます。

Random Forestも基本的にはこれに似ています。

```text
Decision Tree 1 ─┐
Decision Tree 2 ─┤
Decision Tree 3 ─┤
Decision Tree 4 ─┼→ 多数決 → 予測
Decision Tree 5 ─┤
       ...       ─┘
```

**一つ一つの木が少し違う判断をし、それを組み合わせる**わけです。

---

# 🌲 2. なぜ「Random」なのか？

Random Forestでは、単純に同じ決定木を100本コピーするわけではありません。

木ごとに、

* 学習に使うデータ
* 分割に使う特徴量

などにランダム性を入れます。

その結果、

```text
木A → こう判断

木B → 少し違う判断

木C → また違う判断
```

という**多様な木**ができます。

そして、それらを組み合わせます。

---

# 🧠 3. Bagging

Random Forestを理解するうえで、

**Bagging（Bootstrap Aggregating）**

という言葉を覚えておきましょう。

ざっくり言えば、

```text
元データ
 ↓
ランダムにサンプルを作る
 ↓
木1を学習

元データ
 ↓
別のランダムサンプル
 ↓
木2を学習

...
```

とします。

それぞれの木が少し違うデータを見て学習することで、木同士の偏りを減らします。

そして、

```text
木1
木2
木3
...
木N
 ↓
多数決
```

を行います。

---

# 4. 単独の決定木との違い

決定木1本は、

```text
データ
 ↓
1本の判断
```

です。

Random Forestは、

```text
データ
 ↓
複数の判断
 ↓
集約
```

です。

そのため、

> **一つの木が変な方向へ行っても、全体としては安定した予測をしやすくなる**

という利点があります。

---

# 💻 実習1：Irisデータを準備

今回も `iris` を使います。

```python
from sklearn.datasets import load_iris

iris = load_iris()

X = iris.data
y = iris.target
```

train/test分割。

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```

---

# 💻 実習2：決定木1本を作る

まず比較対象を作ります。

```python
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(
    random_state=42
)

tree.fit(
    X_train,
    y_train
)
```

評価。

```python
tree_train_acc = tree.score(
    X_train,
    y_train
)

tree_test_acc = tree.score(
    X_test,
    y_test
)

print(
    "Decision Tree"
)

print(
    "train:",
    tree_train_acc
)

print(
    "test:",
    tree_test_acc
)
```

---

# 💻 実習3：Random Forestを作る

ここから本番。

```python
from sklearn.ensemble import RandomForestClassifier
```

モデル作成。

```python
forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
```

学習。

```python
forest.fit(
    X_train,
    y_train
)
```

評価。

```python
forest_train_acc = forest.score(
    X_train,
    y_train
)

forest_test_acc = forest.score(
    X_test,
    y_test
)
```

表示。

```python
print(
    "Random Forest"
)

print(
    "train:",
    forest_train_acc
)

print(
    "test:",
    forest_test_acc
)
```

---

# 🧠 4. `n_estimators` とは？

ここは重要。

```python
n_estimators=100
```

の意味は、

> **決定木を100本作る**

ということです。

例えば、

```python
n_estimators=10
```

なら10本。

```python
n_estimators=500
```

なら500本。

概念的には、

```text
10本
 ↓
少数の木による多数決

100本
 ↓
より多くの木による多数決

500本
 ↓
さらに多くの木による多数決
```

となります。

---

# 💻 実習4：木の本数を変えてみる

```python
estimators = [
    1,
    5,
    10,
    50,
    100,
    300
]
```

実験。

```python
for n in estimators:

    forest = RandomForestClassifier(
        n_estimators=n,
        random_state=42
    )

    forest.fit(
        X_train,
        y_train
    )

    train_acc = forest.score(
        X_train,
        y_train
    )

    test_acc = forest.score(
        X_test,
        y_test
    )

    print(
        "trees =", n,
        "train =", train_acc,
        "test =", test_acc
    )
```

---

# 👀 観察ポイント

ここでは、

> **木を増やせば必ずtest accuracyが上がるのか？**

を観察してください。

一般に木を増やすことで予測が安定しやすくなりますが、

**100本なら必ず90本より良い**

という単純な話ではありません。

また木を増やせば、

```text
計算量
メモリ使用量
```

も増えます。

AI開発では、

> **性能だけでなく計算コストも設計対象**

です。

---

# 💻 実習5：`max_depth` も変えてみる

第21回で登場した、

```python
max_depth
```

はRandom Forestでも使えます。

```python
forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    random_state=42
)
```

学習。

```python
forest.fit(
    X_train,
    y_train
)
```

評価。

```python
print(
    "train:",
    forest.score(
        X_train,
        y_train
    )
)

print(
    "test:",
    forest.score(
        X_test,
        y_test
    )
)
```

---

# 💻 実習6：木の深さを実験する

```python
depths = [
    1,
    2,
    3,
    5,
    10,
    None
]
```

```python
for depth in depths:

    forest = RandomForestClassifier(
        n_estimators=100,
        max_depth=depth,
        random_state=42
    )

    forest.fit(
        X_train,
        y_train
    )

    train_acc = forest.score(
        X_train,
        y_train
    )

    test_acc = forest.score(
        X_test,
        y_test
    )

    print(
        "depth =", depth,
        "train =", train_acc,
        "test =", test_acc
    )
```

ここで第20回・第21回が戻ってきます。

```text
モデルの複雑さ
       ↓
汎化性能
       ↓
過学習
```

Random Forestでも同じ問題を考える必要があります。

---

# 💻 実習7：特徴量重要度

Random Forestにも、

```python
feature_importances_
```

があります。

```python
importances = forest.feature_importances_

print(
    importances
)
```

特徴量名と一緒に表示しましょう。

```python
for name, importance in zip(
    iris.feature_names,
    importances
):

    print(
        name,
        importance
    )
```

これで、

```text
sepal length
sepal width
petal length
petal width
```

のどれが予測に多く使われているかを確認できます。

---

# 💻 実習8：特徴量重要度を可視化

```python
import matplotlib.pyplot as plt

plt.bar(
    iris.feature_names,
    forest.feature_importances_
)

plt.xticks(
    rotation=45
)

plt.ylabel(
    "Importance"
)

plt.show()
```

これで特徴量重要度をグラフで確認できます。

---

# 🧠 ただし、ここでも注意

第21回と同じですが、

```python
feature_importances_
```

が高い

≠

```text
その特徴量が原因
```

ではありません。

これはあくまで、

> **モデルが予測を行う際に、その特徴量がどれくらい分割に利用されたか**

という情報です。

因果推論とは別物です。

この区別は、将来認知科学や人間-AI研究でデータを扱うときにも重要になる考え方です。

---

# 💻 実習9：確率予測

Random Forestも、

```python
predict_proba()
```

が使えます。

```python
prob = forest.predict_proba(
    X_test
)

print(prob[:5])
```

例えば、

```text
[0.02, 0.95, 0.03]
```

なら、

```text
Class 0 → 2%

Class 1 → 95%

Class 2 → 3%
```

という予測です。

つまりRandom Forestでも、

```text
クラス
+
確率
```

の両方を見ることができます。

---

# 💻 実習10：Cross Validationと組み合わせる

第18回まで戻ります。

```python
from sklearn.model_selection import cross_val_score
```

Random Forestを作って、

```python
forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
```

交差検証。

```python
scores = cross_val_score(
    forest,
    X,
    y,
    cv=5,
    scoring="accuracy"
)
```

表示。

```python
print(
    scores
)

print(
    "mean:",
    scores.mean()
)

print(
    "std:",
    scores.std()
)
```

これで、

> **1回のtrain/test分割だけではなく、複数の分割でRandom Forestの性能を確認する**

ことができます。

---

# ✍️ 演習

今日の `iris` データを使います。

## 問1

Decision Treeを作り、

```text
train accuracy
test accuracy
```

を計算してください。

---

## 問2

Random Forestを作ってください。

条件：

```text
n_estimators=100
random_state=42
```

---

## 問3

Decision TreeとRandom Forestのtest accuracyを比較してください。

---

## 問4

Random Forestの、

```python
feature_importances_
```

を表示してください。

---

## 問5

木の本数を、

```text
1
10
50
100
300
```

と変えて性能を比較してください。

---

## 問6

`max_depth` を、

```text
1
2
3
5
10
None
```

と変えてください。

それぞれ、

```text
train accuracy
test accuracy
```

を記録してください。

---

# 👾 ボス戦：Random Forestの実験設計

ここまでの内容を全部使います。

次のコードを完成させてください。

```python
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=...,
    random_state=42
)

scores = cross_val_score(
    forest,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("scores:", scores)
print("mean:", ...)
print("std:", ...)
```

そして、

```text
max_depth=2
max_depth=3
max_depth=5
max_depth=None
```

について実験してください。

### 最終問題

**「test accuracyが一番高かった設定」と「Cross Validationの平均が一番高かった設定」が違った場合、なぜそんなことが起こり得るでしょうか？**

今日までの、

```text
train/test
Cross Validation
汎化
```

の概念を使って考えてみてください。

---

# 🌱 今日のまとめ

今日の核心は、

> **一つのモデルにすべてを任せるのではなく、複数のモデルの判断を組み合わせることで、より安定した予測を狙う**

という考え方です。

流れは、

```text
Decision Tree
      ↓
複数の木
      ↓
ランダム性を入れる
      ↓
多数決
      ↓
Random Forest
```

です。

そして、

```text
n_estimators
    ↓
木の本数

max_depth
    ↓
各木の複雑さ
```

という2つの重要なハイパーパラメータも扱いました。

---

# 🧭 ここまでのモデル地図

現在、scikit-learnで扱えるモデルがかなり増えてきました。

```text
                    機械学習
                       │
          ┌────────────┴────────────┐
          ↓                         ↓
        回帰                       分類
          │                         │
   LinearRegression        LogisticRegression
                                │
                                ↓
                         Decision Tree
                                │
                                ↓
                         Random Forest
```

そして、どのモデルでも共通して、

```text
データ
 ↓
train/test
 ↓
前処理
 ↓
モデル
 ↓
fit
 ↓
predict
 ↓
評価
 ↓
Cross Validation
 ↓
ハイパーパラメータ調整
```

という**共通の開発ループ**を使えます。

これがかなり重要です。

モデルが変わっても、機械学習エンジニアとしての「型」は変わりません。

---

# 🔜 第23回

## Gradient Boosting：間違いを次のモデルが修正する

次回は、Random Forestとは別系統のアンサンブル学習へ進みます。

Random Forestは、

```text
たくさんの木を
並列的に作る
 ↓
多数決
```

という発想でした。

次の **Gradient Boosting** は、

```text
モデル1
 ↓
間違いを見る
 ↓
モデル2
 ↓
さらに間違いを見る
 ↓
モデル3
 ↓
……
```

という、

> **前のモデルの弱点を、次のモデルが少しずつ修正していく**

という発想です。

ここで、

* Boosting
* Gradient Boosting
* `GradientBoostingClassifier`
* 学習率 `learning_rate`
* 木の数 `n_estimators`
* 木の深さ `max_depth`

を扱います。

Random Forestとはかなり違う「強いモデルの作り方」が見えてくる回です。